# PO Neighborhood Inference On PA Test Plots

This notebook loads a saved `PseudoPA` model checkpoint, gathers surrounding PO surveys around each PA test plot, and predicts the PA test composition from those PO neighborhood inputs.

It is inference-only: no training happens here.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.neighbors import BallTree
from sklearn.neighbors import KNeighborsClassifier

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from geoplant_pseudopa import JointTrainingConfig, evaluate_completion_from_inputs
from geoplant_pseudopa.model import TwoTowerSpeciesModel


In [ ]:
CHECKPOINT_PATH = NOTEBOOK_DIR / "outputs" / "species_only_joint_model.pt"
RAW_PA_TRAIN = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "PA_metadata_train.csv"
RAW_PA_TEST = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "PA_metadata_test.csv"
RAW_PA_TEST_LABELS = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "test_labels.csv"
RAW_PO_TRAIN = REPO_ROOT / "dataset" / "data" / "PresenceOnlyOccurrences" / "PO_metadata_train.csv"

TEST_COUNTRY_FILTER = ["Denmark"]
PO_COUNTRY_REFERENCE_SAMPLES = 200000
PO_COUNTRY_K = 5
PO_NEIGHBOR_RADIUS_KM = 25.0
PO_NEIGHBOR_MIN_SURVEYS = 5
PO_NEIGHBOR_MAX_SURVEYS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_PATH, DEVICE

In [ ]:
def apply_country_filter(raw_frame: pd.DataFrame, country_filter):
    if country_filter is None:
        return raw_frame.reset_index(drop=True)
    if "country" in raw_frame.columns:
        return raw_frame[raw_frame["country"].isin(country_filter)].reset_index(drop=True)
    return raw_frame.reset_index(drop=True)


def infer_po_country_from_pa(pa_reference: pd.DataFrame, po_frame: pd.DataFrame) -> pd.DataFrame:
    reference = pa_reference[["lat", "lon", "country"]].dropna().copy()
    if len(reference) > PO_COUNTRY_REFERENCE_SAMPLES:
        reference = reference.sample(PO_COUNTRY_REFERENCE_SAMPLES, random_state=42)

    knn = KNeighborsClassifier(n_neighbors=PO_COUNTRY_K, weights="distance")
    knn.fit(reference[["lat", "lon"]], reference["country"])

    predicted = po_frame.copy()
    predicted["country"] = knn.predict(predicted[["lat", "lon"]])
    return predicted


def parse_species_string(value: str) -> set[int]:
    stripped = str(value).strip().strip("{}")
    if not stripped:
        return set()
    return {int(token) for token in stripped.split(",") if token}


def build_species_vector(species_ids: set[int], num_species: int) -> torch.Tensor:
    vector = torch.zeros(num_species, dtype=torch.float32)
    if species_ids:
        vector[list(sorted(species_ids))] = 1.0
    return vector


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
checkpoint_config = checkpoint.get("config", {})
config = JointTrainingConfig(num_species=checkpoint_config.get("num_species", 15286))
for key, value in checkpoint_config.items():
    if hasattr(config, key):
        setattr(config, key, value)

model = TwoTowerSpeciesModel(
    num_species=config.num_species,
    embedding_dim=config.embedding_dim,
    hidden_dim=config.hidden_dim,
    dropout=config.dropout,
    use_location=config.use_location,
    location_dim=config.location_dim,
    location_hidden_dim=config.location_hidden_dim,
).to(torch.device(DEVICE))
model.load_state_dict(checkpoint["state_dict"])
model.eval()

pd.Series({
    "checkpoint": str(CHECKPOINT_PATH),
    "device": DEVICE,
    "num_species": config.num_species,
    "embedding_dim": config.embedding_dim,
    "hidden_dim": config.hidden_dim,
})

In [ ]:
raw_pa_reference = pd.read_csv(RAW_PA_TRAIN)
raw_pa_test = apply_country_filter(pd.read_csv(RAW_PA_TEST), TEST_COUNTRY_FILTER)
test_labels = pd.read_csv(RAW_PA_TEST_LABELS)
test_labels = test_labels.rename(columns={"surveyId": "survey_id", "predictions": "species_set"})
test_labels["survey_id"] = test_labels["survey_id"].astype(str)
test_labels["species_set"] = test_labels["species_set"].apply(
    lambda value: "{" + ",".join(token for token in str(value).split() if token) + "}"
)

test_frame = raw_pa_test.rename(columns={"surveyId": "survey_id"}).copy()
test_frame["survey_id"] = test_frame["survey_id"].astype(str)
test_frame = test_frame.merge(test_labels[["survey_id", "species_set"]], on="survey_id", how="inner")
test_frame = test_frame[["survey_id", "lat", "lon", "species_set"]].drop_duplicates().reset_index(drop=True)

test_frame.head()

In [ ]:
po_source = pd.read_csv(RAW_PO_TRAIN)
if TEST_COUNTRY_FILTER is not None:
    po_source = infer_po_country_from_pa(raw_pa_reference, po_source)
    po_source = apply_country_filter(po_source, TEST_COUNTRY_FILTER)

po_surveys = (
    po_source.groupby("surveyId")
    .agg(
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        species_set=("speciesId", lambda col: set(int(species_id) for species_id in col.dropna().astype(int))),
    )
    .reset_index()
    .rename(columns={"surveyId": "survey_id"})
)

pd.DataFrame([
    {"denmark_pa_test_plots": len(test_frame), "denmark_po_surveys": len(po_surveys)}
])

In [ ]:
po_coords_rad = np.deg2rad(po_surveys[["lat", "lon"]].to_numpy())
po_tree = BallTree(po_coords_rad, metric="haversine")
radius_rad = PO_NEIGHBOR_RADIUS_KM / 6371.0

input_vectors = []
target_vectors = []
prediction_rows = []

for row in test_frame.itertuples(index=False):
    query = np.deg2rad([[float(row.lat), float(row.lon)]])
    within_radius = po_tree.query_radius(query, r=radius_rad)[0].tolist()

    if len(within_radius) < PO_NEIGHBOR_MIN_SURVEYS:
        _, knn_idx = po_tree.query(query, k=min(PO_NEIGHBOR_MIN_SURVEYS, len(po_surveys)))
        neighbor_idx = knn_idx[0].tolist()
    else:
        neighbor_idx = within_radius[:PO_NEIGHBOR_MAX_SURVEYS]

    neighbor_species = set()
    for idx in neighbor_idx:
        neighbor_species.update(po_surveys.iloc[idx]["species_set"])

    target_species = parse_species_string(row.species_set)
    input_vectors.append(build_species_vector(neighbor_species, config.num_species))
    target_vectors.append(build_species_vector(target_species, config.num_species))
    prediction_rows.append(
        {
            "survey_id": row.survey_id,
            "po_neighbor_surveys": len(neighbor_idx),
            "po_neighbor_species": len(neighbor_species),
            "pa_species": len(target_species),
        }
    )

neighbor_summary = pd.DataFrame(prediction_rows)
neighbor_summary.describe(include="all")

In [ ]:
po_neighbor_input = torch.stack(input_vectors)
pa_test_target = torch.stack(target_vectors)

test_locations = torch.tensor(test_frame[["lat", "lon"]].to_numpy(dtype=float), dtype=torch.float32)

metrics = evaluate_completion_from_inputs(
    model,
    x_input=po_neighbor_input,
    x_true=pa_test_target,
    input_loc=test_locations,
    device=torch.device(DEVICE),
    ks=(5, 10, 20),
)

pd.Series(metrics, name="value")

In [ ]:
with torch.no_grad():
    logits = model.score_all_species(po_neighbor_input.to(torch.device(DEVICE)))
    probs = torch.sigmoid(logits).cpu()

observed_mask = po_neighbor_input > 0
masked_probs = probs.masked_fill(observed_mask, float("-inf"))
top20 = torch.topk(masked_probs, k=20, dim=1).indices

predictions = neighbor_summary.copy()
predictions["top20_predicted_species"] = [" ".join(str(int(species_id)) for species_id in row.tolist()) for row in top20]
predictions.head()

In [ ]:
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
predictions.to_csv(OUTPUT_DIR / "po_neighborhood_predictions_denmark.csv", index=False)
neighbor_summary.to_csv(OUTPUT_DIR / "po_neighborhood_summary_denmark.csv", index=False)
pd.Series({
    "predictions_csv": str(OUTPUT_DIR / "po_neighborhood_predictions_denmark.csv"),
    "summary_csv": str(OUTPUT_DIR / "po_neighborhood_summary_denmark.csv"),
})